In [ ]:
library(SeuratObject)
library(Seurat)
library(dplyr)
library(SeuratDisk)
library(purrr)
library(tools)
library(lsa)
library(Matrix)
library(FNN)

In [ ]:
python_env <- "/net/data.isilon/ag-saez/yliu/SOFTWARE/.miniconda3/envs/multi2/bin/python"
reticulate::use_python(python_env, required = TRUE)
reticulate::py_config()
torch <- reticulate::import("torch")

In [ ]:
df <- read.csv("marker.csv")[1:500, ]
df

In [ ]:
cts = c('GCBC','NBC_MBC','FDC','epithelial','CD4_T','myeloid') #200，200，200，50，50
donor <-'BCLL-9-T'
file <- paste0('data/spatial/ST_output/', donor, '.rds')
model.est <- readRDS(file)
F_list <- model.est$F_list

In [ ]:
cosine_mat <- function(A, B) {
  A_norm <- sqrt(rowSums(A^2))
  B_norm <- sqrt(rowSums(B^2))
  #print(dim(A))
  #print(dim(B))
  sim <- A %*% t(B)
  sim <- sim / (A_norm %o% B_norm)
  return(sim)
}


# Compute Moran's R weights based on spatial positions
moranR_weights <- function(pos, l = 1) {
  # Convert position matrix to appropriate format
  pos <- as.matrix(pos)
  n <- nrow(pos)
  
  # Calculate pairwise distances
  d <- as.matrix(dist(pos, method = "euclidean"))
  
  # Compute weights using Gaussian kernel
  w <- exp(-d^2 / (2 * l^2))
  diag(w) <- 0
  
  # Normalize weights
  W <- sum(w)
  weight <- (n / W) * w
  
  return(weight)
}

# Compute Moran's R statistic and p-value
# x and y are gene × spot matrices
# Returns gene(x) × gene(y) similarity matrix
moranR <- function(x, y, w) {
  # Get dimensions
  l_s <- nrow(x)
  l_x <- ncol(x)
  l_y <- ncol(y)
  n <- nrow(w)
  
  # Transpose x and y
  x <- t(x)
  y <- t(y)
  
  # Calculate means
  x_mean <- rowMeans(x)
  y_mean <- rowMeans(y)
  
  # Calculate differences from mean
  x_diff <- x - x_mean
  y_diff <- y - y_mean
  
  # Calculate numerator using matrix operations
  # Equivalent to einsum('pi,lj,ij->pl', x_diff, y_diff, w)
  numerator <- x_diff %*% w %*% t(y_diff)
  
  x_norm <- sqrt(rowSums(x_diff^2))
  y_norm <- sqrt(rowSums(y_diff^2))
  denominator <- outer(x_norm, y_norm)
  

  moranR <- numerator / denominator
  
  w_squared_sum <- sum(w * t(w))  
  w_row_sum <- rowSums(w)
  w_col_sum <- colSums(w)
  w_total_sum <- sum(w)
  
  var <- (n^2 * w_squared_sum - 
          2 * n * sum(w_row_sum * w_col_sum) + 
          w_total_sum^2) / (n^2 * (n - 1)^2)
  
  # Calculate z-score
  z <- moranR / sqrt(var)
  
  # Calculate p-value (one-tailed test, upper tail)
  p <- pnorm(z, lower.tail = FALSE)
  

  return(list(moranR = moranR, p = p))
}

topk_matrix <- function(mat, k = 100) {
  mat_top <- matrix(0, nrow = nrow(mat), ncol = ncol(mat))
  
  for(i in 1:nrow(mat)) {
    row <- mat[i, ]
    if(sum(!is.na(row)) > 0){ 
      top_idx <- order(row, decreasing = TRUE)[1:min(k, length(row))]
      mat_top[i, top_idx] <- row[top_idx]
    }
  }
  
  return(mat_top)
}

In [ ]:
coor <- read.csv("data/spatial/coordinates.csv", header = TRUE, stringsAsFactors = FALSE)
coor <- coor[coor$X %in% rownames(model.est$beta), ]
slice <- 'c28w2r_7jne4i'
pos <- coor[startsWith(coor$X, slice), ]
pos <- pos[, c("row", "col")]

In [ ]:
ct_expr <- list()

for(i in seq_along(cts)) {
  ct <- cts[i]
  print(ct)
  if(!ct %in% names(F_list)) next
  mat <- F_list[[ct]]           
  top_genes <- na.omit(df[[ct]])
  top_genes <- top_genes[nzchar(top_genes)]
  #print(top_genes)
  ct_expr[[ct]] <- mat[top_genes, ]
}


out_dir <- "data/spatial/graph/inter/"
if(!dir.exists(out_dir)) dir.create(out_dir, recursive = TRUE)


for(i in 1:(length(cts)-1)) {
  for(j in (i+1):length(cts)) {
    ct1 <- cts[i]
    ct2 <- cts[j]
    w <- moranR_weights(pos)
    if(!(ct1 %in% names(ct_expr)) || !(ct2 %in% names(ct_expr))) next
    
    mat1 <- t(ct_expr[[ct1]])
    mat2 <- t(ct_expr[[ct2]])
    
    #cos_sim <- cosine_mat(mat1, mat2)
    results <- moranR(mat1,mat2,w)
    print(dim(results$moranR))
    #cos_sim <- topk_matrix(results$moranR)
    cos_sim <- results$moranR
    p_mat <- results$p  
    p_adj <- matrix(p.adjust(as.vector(p_mat), method = "BH"),
                    nrow = nrow(p_mat), ncol = ncol(p_mat))
    cos_sim[p_adj >= 0.01| cos_sim < quantile(cos_sim[p_adj < 0.01], 0.95)] <- 0
    print(dim(cos_sim))
    print(sum(cos_sim != 0))
    cos_sim <- torch$from_numpy(as.matrix(cos_sim))  
     
    save_name <- paste0(out_dir, ct1, "_", ct2, ".pt")
    
    torch$save(cos_sim, save_name)
  }
}